<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Classification with Spark MLlib: Decision Trees, Random Forests and Logistic Regression

*Session 5 · Notebook 03.04 · Lecture · Student version*

## Overview

This notebook brings the three classification models from Session 4 to Spark, in the same order: the **decision tree**, a set of readable if-then rules; the **random forest**, many trees averaged for accuracy and stability; and **logistic regression**, the workhorse of scorecards.

We predict **Titanic survival**, in two parts:

- **Part A, trees.** Prepare the data the way trees need it (fill the gaps, turn each category into a number), train a decision tree and a random forest, evaluate them, and see how the same steps fit into a pipeline.
- **Part B, logistic regression.** Add the extra preparation logistic regression needs (one-hot encoding), train it, move the decision threshold, and compare all three models.

## Learning Objectives

By the end of this notebook you will be able to:

- Prepare data for MLlib with `Imputer`, `StringIndexer`, `OneHotEncoder` and `VectorAssembler`, and say which models need which steps.
- Explain the extra output columns of an MLlib classifier: `rawPrediction`, `probability` and `prediction`.
- Evaluate a classifier with a confusion matrix, accuracy, precision, recall, F1 and ROC AUC.
- Fit and interpret a `DecisionTreeClassifier` (its rules) and a `RandomForestClassifier` (its feature importances), first step by step and then as a pipeline.
- Fit a `LogisticRegression`, read its odds ratios, and move the decision threshold.
- Compare models on the same test set and recommend one.

## Prerequisites

- Notebook 03.02 (regression with MLlib: `VectorAssembler`, `StandardScaler`, pipelines, evaluators).
- Session 4 classification notebooks: decision trees, random forests, logistic regression and their evaluation.

## Index

1. [Why this matters](#why)
2. [Setup: data and split](#setup)
3. [What changes for classification in MLlib](#changes)
4. [Evaluating a classifier](#evaluate)

**Part A: Decision trees and random forests**

5. [Preparing the data for the trees](#prep_trees)
6. [Decision trees](#tree)
7. [Random forests](#forest)

**Part B: Logistic regression**

8. [Preparing the data for logistic regression](#prep_lr)
9. [Logistic regression](#logreg)
10. [The decision threshold](#threshold)
11. [Application: choosing a model](#application)

12. [Exercises](#exercises)
13. [Challenge](#challenge)
14. [Key Takeaways](#takeaways)
15. [Conclusion](#conclusion)
16. [Further Reading & Resources](#reading)

<a id="why"></a>
# Section 1: Why this matters

Many of the most important questions in a business are yes/no questions: will this customer default, is this transaction fraudulent, will this client leave? Classification models answer them, and in regulated settings they must also **explain** their answers.

| Model | Strength | Typical use |
|---|---|---|
| Decision tree | Readable if-then rules | Policy rules, segmentation, explaining decisions |
| Random forest | Accurate and stable; ranks the importance of features | Challenger models, when accuracy matters most |
| Logistic regression | Every coefficient converts to an odds ratio: a plain-language, auditable effect | Scorecards, regulatory models, baselines |

Running them in Spark means they can be trained on every account and every transaction, not a sample, and scored over a whole portfolio in one pass.

<a id="setup"></a>
# Section 2: Setup: data and split

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, precision_score, recall_score
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Imputer, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
SEED = 42
from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_03_04_Classification").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# Default: the pandas bridge
titanic = pandas_bridge(spark, "train_titanic")

# Alternative 1: the same in plain code (pandas reads the file, Spark converts it)
# titanic = spark.createDataFrame(pd.read_csv(session_datasets_http["train_titanic"]))

# Alternative 2: Spark reads the file straight from S3
# titanic = spark.read.csv(session_datasets["train_titanic"], header=True, inferSchema=True)

# Keep the label and the seven predictors
data = titanic.select("Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked")

# Split FIRST: every preparation step below learns from train only
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
print("train:", train.count(), "| test:", test.count())
train.show(5)

<a id="changes"></a>
# Section 3: What changes for classification in MLlib

**Definition.** A fitted MLlib classifier's `transform` adds three columns:

| Column | Contains | Used for |
|---|---|---|
| `rawPrediction` | Internal scores (log-odds for logistic regression, counts of training rows in the leaf for a tree) | The AUC evaluator reads it by default |
| `probability` | A vector: [P(class 0), P(class 1)] | Ranking, custom thresholds |
| `prediction` | The predicted class (0.0 or 1.0) using a 0.5 threshold | Confusion matrix, accuracy, precision, recall |

**Analogy.** A weather forecaster who gives both a percentage chance of rain (the probability) and a yes/no "take an umbrella" (the prediction). The yes/no depends on where you draw the line, which is a separate decision from the forecast itself.

**Explanation.** The label must be numeric (0.0/1.0); `Survived` already is. Evaluation uses two evaluators: `BinaryClassificationEvaluator` (AUC, from the scores) and `MulticlassClassificationEvaluator` (accuracy, precision, recall, F1, from the predicted classes). To work with the probability of survival as a number, turn the vector into an array with `vector_to_array` and take element 1.

> **AUC for trees: use the probability column.** For a single decision tree, `rawPrediction` holds the raw **counts** of training rows in the leaf, for example `[95, 39]`. The AUC evaluator ranks rows by the second number, so a big leaf with 39 survivors would outrank a small leaf where every passenger survived, and the AUC comes out far too low. Setting `rawPredictionCol="probability"` ranks by the share instead (39 / 134), which is right for every model in this notebook, so the helpers below always do it.

<a id="evaluate"></a>
# Section 4: Evaluating a classifier

**Definition.** The **confusion matrix** counts the four outcomes: true negatives, false positives, false negatives and true positives. The usual metrics are built from it:

| Metric | Question it answers | From the matrix |
|---|---|---|
| Accuracy | What share of all predictions were right? | (TP + TN) / all |
| Precision (class 1) | Of the passengers predicted to survive, how many did? | TP / (TP + FP) |
| Recall (class 1) | Of the passengers who survived, how many did we find? | TP / (TP + FN) |
| F1 | A balance of precision and recall | harmonic mean |
| ROC AUC | How well does the model **rank** survivors above non-survivors, over all thresholds? | 0.5 = random, 1.0 = perfect |

**Analogy.** A fishing net. Recall is the share of the fish in the lake that you caught; precision is the share of your catch that is actually fish rather than old boots. A bigger net raises recall and lowers precision.

**Explanation.** Accuracy alone can mislead, especially when one class is rare (the German credit data in practical 03.05). Always read precision and recall for the class you care about, and AUC for ranking quality. **Survived = 1 is the positive class** here.

Three helpers are reused for every model: a table of the metrics (computed in Spark), a confusion-matrix plot (as in Session 4: the test predictions are small, so we bring the two columns to pandas and let scikit-learn draw it), and readable feature names from a vector column.

In [ ]:
def scores(pred, label="Survived", name="model"):
    """The usual classification metrics for one set of predictions."""
    mc = lambda metric, **kw: MulticlassClassificationEvaluator(labelCol=label, metricName=metric, **kw).evaluate(pred)
    return {
        "model": name,
        "accuracy": mc("accuracy"),
        "precision_1": mc("precisionByLabel", metricLabel=1.0),
        "recall_1": mc("recallByLabel", metricLabel=1.0),
        "f1": mc("f1"),
        # AUC from the probability column (see Section 3): right for trees, forests and logistic regression alike
        "auc": BinaryClassificationEvaluator(labelCol=label, rawPredictionCol="probability",
                                             metricName="areaUnderROC").evaluate(pred),
    }

def plot_confusion(pred, title, label="Survived"):
    """Confusion matrix plot: rows = actual class, columns = predicted class."""
    pdf = pred.select(label, "prediction").toPandas()
    ConfusionMatrixDisplay.from_predictions(pdf[label], pdf["prediction"].astype(int),
                                            display_labels=["died", "survived"], cmap="Blues")
    plt.title(title); plt.show()

def feature_names(df, col="features"):
    """Names of the positions of an assembled vector column, in order."""
    attrs = df.schema[col].metadata["ml_attr"]["attrs"]
    return [a["name"] for a in sorted((a for g in attrs.values() for a in g), key=lambda a: a["idx"])]

def show_scores(s):
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in s.items()})

# Part A: Decision trees and random forests

<a id="prep_trees"></a>
# Section 5: Preparing the data for the trees

MLlib models need **one numeric vector column** with **no missing values**. The Titanic data has gaps and text categories, so it needs some work first. For the trees, four steps are enough:

| Step | Column(s) | Problem | What we do | Tool |
|---|---|---|---|---|
| 1 | `Embarked` | A few missing values (text) | Fill with the most common port in the training data | `na.fill` |
| 2 | `Age` | Many missing values (numeric) | Fill with the median age of the training data | `Imputer` |
| 3 | `Sex`, `Pclass`, `Embarked` | Categories, not numbers | Give each category a number | `StringIndexer` |
| 4 | all features | MLlib reads one vector column | Pack everything into `features` | `VectorAssembler` |

**Why is a number per category enough for a tree?** A tree only asks questions such as "is `Sex_idx` 0?" or "is `Pclass_idx` in {0, 1}?". It never adds or multiplies the numbers, so it does not matter that "female = 1" is not "more" than "male = 0". (Logistic regression does multiply them, which is why Part B adds one more step.)

**The golden rule:** every value we fill in, and every category list, is learned from **`train` only** and then applied to both `train` and `test`, so nothing from the test set leaks into the model.

**Step 1: fill the missing ports.** Only a few values are missing, so the simplest sensible choice is the most common port, found in the training data.

In [ ]:
# Count the missing values per column, before any filling
# Most common port IN THE TRAINING DATA: group, count, sort descending, take the first row
# Fill the gaps in both parts with that same value

**Step 2: fill the missing ages with `Imputer`.** `Imputer` is an estimator: `fit` learns the median age from the training data, `transform` writes a new column `Age_imp` with the gaps filled. The original `Age` column is kept, so you can compare the two.

In [ ]:
# Rows where Age was missing: Age_imp now holds the median

**Step 3: number the categories with `StringIndexer`.** It gives each category of a column a number: the most frequent category gets 0.0, the next 1.0, and so on. It is an estimator: `fit` learns the list of categories from the training data. `Pclass` is stored as a number already, but it is really a category (first, second, third class), so we index it too.

In [ ]:
# Your turn. Write your solution here:

**Step 4: pack everything into one vector with `VectorAssembler`.** It is a transformer (nothing to learn): it takes a list of numeric columns and writes one `features` column.

In [ ]:
# Your turn. Write your solution here:

<a id="tree"></a>
# Section 6: Decision trees

**Definition.** A **decision tree** splits the data again and again on the feature and cut-off that best separate the classes, ending in leaves that predict a class. The path from the root to a leaf is an if-then rule.

**Example.** "If Sex is male and Age > 9.5, predict did not survive."

**Analogy.** The game of twenty questions: each question is chosen to split the remaining possibilities as cleanly as possible.

**Explanation.** The data is ready (`train_tree`, `test_tree`), so training is one line: `fit` on the training data, `transform` the test data. `maxDepth` limits how many questions in a row the tree may ask: deeper trees fit the training data better but **overfit** more. MLlib prints the fitted rules with `toDebugString`; feature numbers in that text refer to positions in the feature vector.

In [ ]:
# Your turn. Write your solution here:

In [ ]:
## Visualising the tree
print("feature positions:", dict(enumerate(tree_features)))
print("\n".join(tree_fitted.toDebugString.splitlines()[:16]))

**Reading the rules.** `feature 4` is `Sex_idx`; `male` is the more frequent category, so `male = 0.0` and `female = 1.0`, and `feature 4 in {0.0}` means "male". The first split is on sex, exactly as the survival rates by sex suggest (about 74% of women survived against 19% of men); further splits use class, age and fare.

In [ ]:

plot_confusion(tree_pred, "Decision tree confusion matrix (test set)")
tree_scores = scores(tree_pred, name="Decision tree (depth 4)")
show_scores(tree_scores)

In [ ]:
# Your turn. Write your solution here:

In [ ]:
tree_importance = pd.Series(tree_fitted.featureImportances.toArray(), index=tree_features).sort_values()
tree_importance.plot(kind="barh", figsize=(8, 4), color="steelblue")
plt.title("Decision tree feature importance"); plt.xlabel("relative importance"); plt.show()

**The same steps as a pipeline.** We fitted the imputer, fitted the indexer, ran the assembler, then fitted the tree, each time remembering to apply the same step to `test`. A `Pipeline` (notebook 03.02, Section 12) does exactly this in one object: give it the **unfitted** steps in order, call `fit` once on the raw training data, then `transform` the raw test data.

Step 1 (filling `Embarked`) stays outside the pipeline: it was a one-off fill, already applied to `train` and `test`.

In [ ]:
# Your turn. Write your solution here:

Same model, same result, but one `fit` and one `transform` instead of a separate call for every step, and no way to forget a step on the test data.

<a id="forest"></a>
# Section 7: Random forests

**Definition.** A **random forest** trains many decision trees, each on a random sample of the rows and choosing among a random subset of features at each split, and combines their votes.

**Example.** 100 trees, each slightly different; the forest's probability of survival is the average of the trees' votes.

**Analogy.** Asking a hundred independent advisers instead of one: individual advisers make idiosyncratic mistakes, but their average opinion is much more reliable.

**Explanation.** Averaging reduces the variance of a single deep tree, so forests are usually more accurate and stable. The price is direct readability, partly recovered with **feature importances**: how much each feature reduced impurity across all the trees. The forest uses the same prepared data as the tree. (On serverless compute a single model is limited to 100 MB, which a forest of this size is far below.)

In [ ]:
# Your turn. Write your solution here:

In [ ]:
# Your turn. Write your solution here:

In [ ]:
forest_importance = pd.Series(forest_fitted.featureImportances.toArray(), index=tree_features).sort_values()
forest_importance.plot(kind="barh", figsize=(8, 4), color="seagreen")
plt.title("Random forest feature importance"); plt.xlabel("relative importance"); plt.show()

Sex, fare, class and age carry most of the importance. The forest spreads importance more evenly than the single tree, because its many trees use different features.

# Part B: Logistic regression

<a id="prep_lr"></a>
# Section 8: Preparing the data for logistic regression

You know the model from Session 4: it multiplies each input by a coefficient, adds them up and turns the total into a probability. **That is why the category numbers from Section 5 are not enough.** With `Pclass_idx` = 0, 1, 2 the model would treat the classes as a scale, as if one class were "twice" another. The fix is **one-hot encoding**: one 0/1 column per category, so each category gets its own coefficient.

**One-hot encode with `OneHotEncoder`.** It turns each index column into a 0/1 vector. Spark prints these as **sparse** vectors: `(2,[0],[1.0])` means "length 2, value 1.0 at position 0, zeros elsewhere"; `(2,[],[])` is all zeros.

**The baseline.** The encoder **drops the last category** of each column (`dropLast=True`), the same "avoid the dummy trap" idea as `drop="first"` in scikit-learn. The dropped category is shown by all zeros and becomes the **baseline** the others are compared with. Because `StringIndexer` numbers the categories from most to least frequent, the baseline is the **least frequent** category of each column.

We start from the tree data of Section 5 (gaps filled, categories indexed), so only two steps are new: encode, then assemble the vector for logistic regression.

In [ ]:
ohe_cols = [c + "_ohe" for c in cat_cols]               # Sex_ohe, Pclass_ohe, Embarked_ohe

encoder = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols)
encoder_model = encoder.fit(train_tree)                  # learns how many categories each column has
train_lr = encoder_model.transform(train_tree)
test_lr = encoder_model.transform(test_tree)
train_lr.select("Sex", "Sex_idx", "Sex_ohe", "Pclass", "Pclass_idx", "Pclass_ohe").distinct().show(6, truncate=False)

# The baseline of each column is its last (least frequent) category
baselines = {col: labels[-1] for col, labels in zip(cat_cols, indexer_model.labelsArray)}
print("baselines:", baselines)

In [ ]:
logreg_features = num_cols + ohe_cols                     # 4 numeric columns + the one-hot vectors

lr_assembler = VectorAssembler(inputCols=logreg_features, outputCol="features_lr")
train_lr = lr_assembler.transform(train_lr)
test_lr = lr_assembler.transform(test_lr)
train_lr.select("Sex", "Pclass", "Embarked", "features_lr", "Survived").show(3, truncate=False)

<a id="logreg"></a>
# Section 9: Logistic regression

Here we focus on how to run it in MLlib. Training is the same pattern as the trees: `fit` on `train_lr`, `transform` `test_lr`.

**No separate scaler is needed.** `LogisticRegression` standardises internally while fitting but reports the coefficients in the **original units**, so the `Age` odds ratio reads "per year". (Exercise 4 standardises the numeric features yourself, to see what changes.)

In [ ]:
logreg = LogisticRegression(featuresCol="features_lr", labelCol="Survived", maxIter=100)
logreg_fitted = logreg.fit(train_lr)
lr_pred = logreg_fitted.transform(test_lr)

# The probability of survival as a number, next to the 0/1 prediction (threshold 0.5)
(lr_pred.withColumn("p_survive", F.round(vector_to_array("probability")[1], 3))
        .select("Sex", "Pclass", "Age", "Fare", "Survived", "p_survive", "prediction")
        .show(5))

In [ ]:
plot_confusion(lr_pred, "Logistic regression confusion matrix (threshold 0.5)")
lr_scores = scores(lr_pred, name="Logistic regression")
show_scores(lr_scores)

In [ ]:
# Odds ratios: exp(coefficient). Above 1 raises the odds of survival, below 1 lowers them.
names = feature_names(lr_pred, "features_lr")
coefs = logreg_fitted.coefficients.toArray()
odds = pd.DataFrame({"feature": names, "coefficient": coefs, "odds_ratio": np.exp(coefs)}).sort_values("odds_ratio")
print("baselines:", baselines)
print(odds.round(3).to_string(index=False))

**Reading it.** `Sex_ohe_male` has an odds ratio far below 1: a man's odds of survival were a small fraction of a woman's (the baseline) with the same class, age, fare and family. Each class column compares that class with the baseline class printed above. These are the statements a stakeholder or a regulator can check: a driver, its direction and its size, against an explicit reference group.

<a id="threshold"></a>
# Section 10: The decision threshold

**Definition.** The **threshold** is the probability above which a case is labelled positive. MLlib uses 0.5 by default; you can set `threshold` on `LogisticRegression`, or apply your own cut-off to the probability column.

**Example.** Lowering the cut-off to 0.3 labels more passengers as survivors: more true survivors are found (recall up), at the cost of more wrong calls (precision down).

**Analogy.** The sensitivity of a smoke alarm. Set it high and it catches every fire but also burnt toast; set it low and it stays quiet through the toast but may miss a slow fire.

**Explanation.** The threshold is a **business decision**, set by the relative cost of the two errors: a missed defaulter versus a declined good customer, a missed fraud versus a blocked genuine payment. The model gives probabilities; the business chooses the line.

In [ ]:
# Your turn. Write your solution here:

The same trade-off over every threshold, as a chart. The test predictions are small, so we bring the label and the probability to pandas.

In [ ]:
p1 = vector_to_array("probability")[1]          # probability of survival as a number
rows = []
for t in [0.5, 0.4, 0.3]:
    labelled = lr_pred.withColumn("prediction", F.when(p1 >= t, 1.0).otherwise(0.0))   # re-label with cut-off t
    s = scores(labelled, name=f"threshold {t}")
    rows.append((t, round(s["recall_1"], 3), round(s["precision_1"], 3), round(s["accuracy"], 3)))
pd.DataFrame(rows, columns=["threshold", "recall_survived", "precision_survived", "accuracy"])

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, precision, marker="o", label="precision (survived)")
plt.plot(thresholds, recall, marker="o", label="recall (survived)")
plt.axvline(0.5, color="grey", ls=":", label="default threshold 0.5")
plt.xlabel("threshold"); plt.ylabel("score"); plt.title("Moving the threshold trades recall for precision")
plt.legend(); plt.show()

<a id="application"></a>
# Section 11: Application: choosing a model

**Scenario.** Recommend one model for predicting survival, using the same test set for all three.

In [ ]:
comparison = pd.DataFrame([tree_scores, forest_scores, lr_scores]).set_index("model").round(3)
comparison[["auc", "accuracy", "precision_1", "recall_1", "f1"]]

**The ROC curve** plots the true positive rate (recall) against the false positive rate as the threshold moves from 1 to 0. A curve closer to the top-left corner ranks better; the area under it is the AUC in the table.

In [ ]:
plt.figure(figsize=(6.5, 5.5))
for name, pred, s in [("Decision tree", tree_pred, tree_scores), ("Random forest", forest_pred, forest_scores),
                      ("Logistic regression", lr_pred, lr_scores)]:
    roc_pd = pred.select("Survived", vector_to_array("probability")[1].alias("p")).toPandas()
    fpr, tpr, _ = roc_curve(roc_pd["Survived"], roc_pd["p"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {s['auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random")
plt.xlabel("false positive rate"); plt.ylabel("true positive rate (recall)"); plt.title("ROC curves, test set")
plt.legend(); plt.show()

**Recommendation.** On data this size the three models are usually close. The forest tends to rank best (AUC), the tree is the easiest to explain as rules, and logistic regression combines competitive accuracy with odds ratios that state each effect and its size. Where decisions must be justified line by line, logistic regression is the natural first choice, with the forest as a challenger that shows how much accuracy (if any) the simpler model leaves on the table.

<a id="exercises"></a>
# Exercises

### Exercise 1: Depth and overfitting

Fit decision trees on `train_tree` with `maxDepth` 2, 4, 8 and 12. For each, print the AUC on the **training** data and on the **test** data (`test_tree`). What happens as depth grows?

In [ ]:
# Your turn

### Exercise 2: Most and least likely survivors

Using `lr_pred`, add the probability of survival as a column `p_survive` (rounded to 3 decimals). Show the five test passengers with the **highest** `p_survive` and the five with the **lowest**, with `Sex`, `Pclass`, `Age`, `Fare` and `Survived`. Does the pattern make sense?

> *Hint:* `vector_to_array("probability")[1]` turns the probability vector into a number, as in Section 9; then `orderBy` and `show(5)`.

In [ ]:
# Your turn

### Exercise 3: Use the threshold parameter

Fit a second logistic regression on `train_lr` with `threshold=0.35`, and compare its confusion matrix on `test_lr` with the default model's.

In [ ]:
# Your turn

### Exercise 4: Standardise the numeric features

`Age_imp` and `Fare` are on very different scales (years against pounds). Standardise the four numeric columns, then refit the logistic regression:

1. Assemble `num_cols` into a vector `num_vec` on `train_tree` and `test_tree`.
2. Fit a `StandardScaler` (`withMean=True`) on the training part to get `num_scaled`, and apply it to both parts.
3. One-hot encode as in Section 8, assemble `num_scaled` plus the one-hot columns into `features_std`, and fit `LogisticRegression` on it.

Compare the test AUC and the odds ratios of the numeric features with the original model. What changed, and what did not?

> *Hint:* `StandardScaler` works on a vector column, so assemble the numeric columns first.

In [ ]:
# Your turn

<a id="challenge"></a>
# Challenge

### Challenge: Engineer family features

Add `FamilySize = SibSp + Parch + 1` and `IsAlone` (1 if FamilySize is 1, else 0) to `train_lr` and `test_lr`, assemble them with `Age_imp`, `Fare` and the one-hot columns (replacing `SibSp` and `Parch`), refit the logistic regression, and compare the test AUC and the new odds ratios with the original model.

In [ ]:
# Your turn

<a id="takeaways"></a>
## Key Takeaways

| Task | MLlib |
|---|---|
| Fill numeric gaps | `Imputer(strategy="median")`, fitted on train |
| Number categories | `StringIndexer`; most frequent = 0.0; `labelsArray` shows the order |
| Trees | Category numbers are enough: trees only ask "is it in this group?" |
| Logistic regression | Needs `OneHotEncoder`; the dropped (least frequent) category is the baseline |
| One vector column | `VectorAssembler(inputCols=..., outputCol=...)` |
| Pipeline | `Pipeline(stages=[imputer, indexer, assembler, model])`: one `fit`, one `transform` |
| Classifier outputs | `rawPrediction`, `probability` (vector), `prediction` (0.5 threshold) |
| Probability as a number | `vector_to_array("probability")[1]` |
| Accuracy, precision, recall, F1 | `MulticlassClassificationEvaluator` (`precisionByLabel`, `recallByLabel` with `metricLabel=1.0`) |
| AUC | `BinaryClassificationEvaluator(rawPredictionCol="probability", metricName="areaUnderROC")` |
| Decision tree | `DecisionTreeClassifier(maxDepth=...)`; rules via `toDebugString` |
| Random forest | `RandomForestClassifier(numTrees=..., maxDepth=...)`; `featureImportances` |
| Odds ratios | `np.exp(model.coefficients)` |
| Threshold | `threshold=` on the model, or a cut-off on the probability |

<a id="conclusion"></a>
## Conclusion

You can now prepare data for MLlib step by step, knowing which models need which steps, wrap those steps in a pipeline, and build, evaluate and explain the three core classification models in Spark: a decision tree with its rules, a random forest with feature importances, and logistic regression with odds ratios, moving the decision threshold deliberately. The practical (03.05) applies them to German credit data, where class imbalance makes the choice of metric and threshold critical.

<a id="reading"></a>
## Further Reading & Resources

- Apache Spark: [Classification and regression in MLlib](https://spark.apache.org/docs/latest/ml-classification-regression.html)
- Apache Spark: [Extracting, transforming and selecting features](https://spark.apache.org/docs/latest/ml-features.html)
- Apache Spark: [LogisticRegression API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html)
- Apache Spark: [RandomForestClassifier API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.RandomForestClassifier.html)
- Apache Spark: [MulticlassClassificationEvaluator API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.MulticlassClassificationEvaluator.html)